In [1]:
# Stage 1 — LOAD

!pip install transformers -q
import pandas as pd
from transformers import pipeline


# Writing .csv file

csv_text = """id,review
1,"I loved the food and the service was great"
2,"Terrible experience, very slow and rude"
3,"It was fine, nothing special"
4,"Absolutely fantastic, will come again"
5,"Not worth the money at all"
"""

with open("reviews.csv", "w") as f:
    f.write(csv_text)
print('file_written')

file_written


In [2]:
# Stage 1 — LOAD

# Reading reviews.csv

df = pd.read_csv("reviews.csv")
df

,id,review
0,1,I loved the food and the service was great
1,2,"Terrible experience, very slow and rude"
2,3,"It was fine, nothing special"
3,4,"Absolutely fantastic, will come again"
4,5,Not worth the money at all


In [3]:
# Stage 1 — LOAD

# Performing First look ritual

df.head()
df.info()
df.describe()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      5 non-null      int64 
 1   review  5 non-null      object
dtypes: int64(1), object(1)
memory usage: 212.0+ bytes


,id
count,5.000000
mean,3.000000
std,1.581139
min,1.000000
25%,2.000000
50%,3.000000
75%,4.000000
max,5.000000


In [4]:
# STAGE-2

# TODO: remove rows where the review is missing (assign the result back to df)

df = df.dropna(subset=["review"])
print(df.isnull().sum())  # NO MISSING ROWS

#TODO: strip whitespace from the review column (assign it back to df["review"])

df["review"] = df["review"].str.strip()

# TODO: print how many reviews are left, e.g. "Reviews to analyze: 5"
print(f'Reviews to analyze: {len(df)}')

id        0
review    0
dtype: int64
Reviews to analyze: 5


In [5]:
# STAGE-3

# TODO: load the sentiment model into a variable called classifier
classifier = pipeline("sentiment-analysis")

# Create two empty lists to collect results
labels = []
scores = []

# TODO: loop over each review in df["review"]
# - run the classifier on it and grab the result dictionary with [0]
# - append the label to labels
# - append the rounded score to scores

for review in df["review"]:
    result = classifier(review)[0]
    labels.append(result["label"])
    scores.append(round(result["score"], 2))

# TODO: add the two lists to df as new columns: "sentiment" and "confidence"
df["sentiment"] = labels
df["confidence"] = scores

# TODO: print df to see reviews next to their sentiment
df

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

,id,review,sentiment,confidence
0,1,I loved the food and the service was great,POSITIVE,1.00
1,2,"Terrible experience, very slow and rude",NEGATIVE,1.00
2,3,"It was fine, nothing special",NEGATIVE,0.96
3,4,"Absolutely fantastic, will come again",POSITIVE,1.00
4,5,Not worth the money at all,NEGATIVE,1.00


In [6]:
# STAGE-4

# TODO: count positive reviews, negative reviews, and the total

positive = (df["sentiment"] == "POSITIVE").sum()
negative = (df["sentiment"] == "NEGATIVE").sum()
total = len(df)

# TODO: print a report header, the total, and positive/negative counts with percentages

header = "==== Sentiment Analysis Report ===="
print(header)
print(f'Total reviews analyzed:: {total} \n')
print(f'Positive: {positive} ({positive/total*100:.0f}%)')
print(f'Negative: {negative} ({negative/total*100:.0f}%)')

# TODO: find the positive rows, sort by confidence (highest first),
# and print the review text of the top one

positive_df = df[df["sentiment"] == "POSITIVE"]
sorted_df = positive_df.sort_values("confidence", ascending=False)
top_review = sorted_df.iloc[0]["review"]
print(f'\nMost confident positive review\n{top_review}')


==== Sentiment Analysis Report ====
Total reviews analyzed:: 5 

Positive: 2 (40%)
Negative: 3 (60%)

Most confident positive review
I loved the food and the service was great


In [7]:
# STAGE-5

# TODO: save df to a file called reviews_with_sentiment.csv (no index column)

df.to_csv("reviews_with_sentiment.csv", index=False)
print("file_written")

file_written
